# Shopora — train your own watch try-on model (SDXL LoRA)

Fine-tunes a **LoRA** for **Stable Diffusion XL** on your own images. The LoRA is then used with
SDXL **inpainting** (see `serve_tryon.ipynb`) to repaint only the wrist area of a customer's photo,
starting from Shopora's 3D AR render, so the watch looks really worn.

**Licence:** SDXL base and SDXL inpainting are released under CreativeML OpenRAIL++-M, which allows
commercial use (read its use restrictions). Only train on images you have the rights to use.

**Runtime:** `Runtime → Change runtime type → GPU` (T4 works; L4/A100 on Colab Pro is faster).

Two modes:
- **`realism`** — one model for the whole store. Train on many photos of people wearing watches
  (with captions). Teaches realistic skin contact, shadows and lighting.
- **`product`** — one model for one specific watch. Train on 15–30 photos of that watch.
  Teaches its exact design, triggered by the word in `TRIGGER`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install

Versions are pinned to match the training script downloaded below.

In [ ]:
!pip -q install diffusers==0.31.0 transformers==4.46.3 accelerate==1.1.1 peft==0.13.2 \
    bitsandbytes datasets ftfy safetensors
!wget -q -O train_dreambooth_lora_sdxl.py \
    https://raw.githubusercontent.com/huggingface/diffusers/v0.31.0/examples/dreambooth/train_dreambooth_lora_sdxl.py
!accelerate config default

## 2. Configure

Put your images in Google Drive first (see `README.md` for what makes a good dataset):

```
MyDrive/shopora-tryon/datasets/<DATASET_NAME>/   ← images (.jpg/.png/.webp)
                                                 ← captions.csv (optional, realism mode)
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch

MODE = 'realism'            # 'realism' or 'product'
DATASET_NAME = 'wrist-watches'
TRIGGER = 'shpwtch'         # product mode: the made-up word that means "this exact watch"

ROOT = '/content/drive/MyDrive/shopora-tryon'
RAW_DIR = f'{ROOT}/datasets/{DATASET_NAME}'
TRAIN_DIR = f'/content/train/{DATASET_NAME}'
OUTPUT_DIR = f'{ROOT}/loras/{DATASET_NAME}'

gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
supports_bf16 = torch.cuda.get_device_capability(0)[0] >= 8
RESOLUTION = 1024 if gpu_memory_gb >= 20 else 768
PRECISION = 'bf16' if supports_bf16 else 'fp16'
print(f'GPU memory {gpu_memory_gb:.0f} GB → resolution {RESOLUTION}, precision {PRECISION}')

DEFAULT_CAPTION = (
    f'a photo of a wrist wearing a {TRIGGER} watch' if MODE == 'product'
    else "a realistic photo of a person's wrist wearing a watch, natural skin, real lighting"
)

## 3. Prepare the dataset

Checks every image, converts it to RGB, limits it to 1536 px and copies it to fast local disk.
In realism mode, captions come from `captions.csv` (`file_name,text`) when present; otherwise
every image uses `DEFAULT_CAPTION`.

In [ ]:
import csv, json, os, shutil
from pathlib import Path
from PIL import Image, ImageOps

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
shutil.rmtree(TRAIN_DIR, ignore_errors=True)
os.makedirs(TRAIN_DIR, exist_ok=True)

captions = {}
captions_csv = Path(RAW_DIR) / 'captions.csv'
if captions_csv.exists():
    with open(captions_csv, newline='', encoding='utf-8') as f:
        captions = {row['file_name']: row['text'].strip() for row in csv.DictReader(f)}

kept, skipped, metadata = 0, [], []
for path in sorted(Path(RAW_DIR).iterdir()):
    if path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue
    try:
        image = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    except Exception as err:
        skipped.append(f'{path.name}: {err}')
        continue
    if min(image.size) < 512:
        skipped.append(f'{path.name}: smaller than 512 px')
        continue
    image.thumbnail((1536, 1536), Image.LANCZOS)
    name = f'{kept:04d}.jpg'
    image.save(Path(TRAIN_DIR) / name, quality=95)
    metadata.append({'file_name': name, 'text': captions.get(path.name, DEFAULT_CAPTION)})
    kept += 1

if MODE == 'realism':
    with open(Path(TRAIN_DIR) / 'metadata.jsonl', 'w', encoding='utf-8') as f:
        for row in metadata:
            f.write(json.dumps(row) + '\n')

print(f'{kept} images ready, {len(skipped)} skipped')
for reason in skipped:
    print('  skipped', reason)
minimum = 15 if MODE == 'product' else 100
if kept < minimum:
    print(f'⚠️  {MODE} mode works best with at least {minimum} images.')

## 4. Train

Product mode: ~100 steps per image (capped 600–1500). Realism mode: 2000 steps to start.
Checkpoints are saved to Drive every 250 steps, so a disconnected session can be resumed with
`--resume_from_checkpoint=latest`.

In [ ]:
MAX_STEPS = min(1500, max(600, kept * 100)) if MODE == 'product' else 2000

data_args = (
    f'--instance_data_dir="{TRAIN_DIR}" --instance_prompt="{DEFAULT_CAPTION}"' if MODE == 'product'
    else f'--dataset_name="{TRAIN_DIR}" --image_column=image --caption_column=text --instance_prompt="{DEFAULT_CAPTION}"'
)

command = f'''accelerate launch train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0 \
  --pretrained_vae_model_name_or_path=madebyollin/sdxl-vae-fp16-fix \
  {data_args} \
  --output_dir="{OUTPUT_DIR}" \
  --resolution={RESOLUTION} --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 --gradient_checkpointing \
  --use_8bit_adam --mixed_precision={PRECISION} \
  --learning_rate=1e-4 --lr_scheduler=constant --lr_warmup_steps=0 --snr_gamma=5.0 \
  --rank=16 --max_train_steps={MAX_STEPS} --checkpointing_steps=250 --seed=0'''
print(command)
!{command}

## 5. Check the result (text to image)

The base model plus your LoRA, no photo involved. In product mode the watch should resemble yours;
in realism mode wrists should look natural.

In [ ]:
from diffusers import AutoencoderKL, AutoPipelineForText2Image
from IPython.display import display

vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', vae=vae, torch_dtype=torch.float16, variant='fp16'
).to('cuda')
pipe.load_lora_weights(OUTPUT_DIR, weight_name='pytorch_lora_weights.safetensors')

for seed in range(3):
    image = pipe(
        prompt=DEFAULT_CAPTION + ', close-up, sharp focus',
        negative_prompt='blurry, deformed hand, extra fingers, cartoon, render, cgi',
        num_inference_steps=30, guidance_scale=6.0,
        generator=torch.Generator('cuda').manual_seed(seed),
    ).images[0]
    display(image.resize((512, 512)))

del pipe
torch.cuda.empty_cache()

## 6. Next

Open `serve_tryon.ipynb` with `LORA_DIR` set to the `OUTPUT_DIR` above to test inpainting on real
photos and expose the model to Shopora.